# Dynamic Horizon Experiments

This notebook is the runbook for evaluating the horizon-and-distribution-aware TD-MPC2 variant on four DMControl environments. The environments are ordered by increasing expected difficulty for a limited-time single RTX 4070 8GB run.

1. `cartpole-swingup`
2. `finger-spin`
3. `cheetah-run`
4. `walker-walk`

All setup lives in `tdmpc2/configs/`. Commands are meant to be run from `tdmpc2/`.


## Method Script: Online Planning

- Let's first look at the online planning module at a specific time step `t`. In standard TD-MPC2, the planner evaluates candidate action sequences with a fixed rollout horizon `horizon`. In our current implementation, when `adaptive_horizon=true`, the planner rolls each candidate sequence out to `h_max`, but scores it using an effective horizon `H_t` selected dynamically from the candidate depths within `[h_min, h_max]`.
- Concretely, during the same latent rollout we record bootstrapped return estimates at depths `1`, `3`, `5`, and `h_max` whenever those depths are available. Each estimate is the predicted reward accumulated up to that depth plus a policy/Q bootstrap from the latent state at that depth.
- We then compute a model-value inconsistency proxy by comparing these recorded returns. The selector measures each depth's absolute deviation from the mean recorded return estimate and chooses the depth with the smallest inconsistency. The selected `H_t` is used as the return depth for MPPI scoring, and the same adaptive choice is also used during training to limit the model losses and policy update horizon.
- One implementation detail to state clearly: the current code does not use `inconsistency_threshold` or `inconsistency_patience` to truncate at the first sharp disagreement. Those config fields are present as placeholders, while the active selector is currently an argmin over the recorded depth inconsistencies.


## Experiment Matrix

| Order | Environment | Variant | Config file | Hypothesis | Primary metric |
|---:|---|---|---|---|---|
| 1 | `cartpole-swingup` | baseline | `cartpole_swingup_baseline.yaml` | Stable fixed-horizon reference. | `eval/episode_reward` |
| 1 | `cartpole-swingup` | adaptive only | `cartpole_swingup_adaptive_only.yaml` | Uses shorter horizons when value/model disagreement is high. | `eval/episode_reward`, `train/rollout_horizon` |
| 1 | `cartpole-swingup` | behavior only | `cartpole_swingup_behavior_only.yaml` | Reduces OOD policy drift without changing rollout depth. | `eval/episode_reward`, `train/behavior_reg_loss` |
| 1 | `cartpole-swingup` | full | `cartpole_swingup_full.yaml` | Best sample-efficiency and least horizon collapse. | `eval/episode_reward` |
| 2 | `finger-spin` | baseline | `finger_spin_baseline.yaml` | Fixed horizon can be brittle under fast contact dynamics. | `eval/episode_reward` |
| 2 | `finger-spin` | adaptive only | `finger_spin_adaptive_only.yaml` | Dynamic horizon should avoid unreliable long rollouts early. | `eval/episode_reward`, `train/model_value_inconsistency` |
| 2 | `finger-spin` | behavior only | `finger_spin_behavior_only.yaml` | Regularization should reduce exploitative actions. | `eval/episode_reward`, `train/value_overestimation_proxy` |
| 2 | `finger-spin` | full | `finger_spin_full.yaml` | Should combine safer actions with adaptive rollout depth. | `eval/episode_reward` |
| 3 | `cheetah-run` | baseline | `cheetah_run_baseline.yaml` | Strong fixed-horizon locomotion baseline. | `eval/episode_reward` |
| 3 | `cheetah-run` | adaptive only | `cheetah_run_adaptive_only.yaml` | Should improve unstable phases but may still over-query OOD values. | `eval/episode_reward`, `train/horizon_at_h_min` |
| 3 | `cheetah-run` | behavior only | `cheetah_run_behavior_only.yaml` | Should reduce value exploitation, with fixed-horizon cost. | `eval/episode_reward`, `train/value_overestimation_proxy` |
| 3 | `cheetah-run` | full | `cheetah_run_full.yaml` | Expected best reward/cost tradeoff. | `eval/episode_reward` |
| 4 | `walker-walk` | baseline | `walker_walk_baseline.yaml` | Hardest fixed-horizon locomotion reference. | `eval/episode_reward` |
| 4 | `walker-walk` | adaptive only | `walker_walk_adaptive_only.yaml` | Should adapt during unstable gait learning. | `eval/episode_reward`, `train/rollout_horizon` |
| 4 | `walker-walk` | behavior only | `walker_walk_behavior_only.yaml` | Should reduce OOD value exploitation during gait search. | `eval/episode_reward`, `train/value_overestimation_proxy` |
| 4 | `walker-walk` | full | `walker_walk_full.yaml` | Expected best final and early reward under time limits. | `eval/episode_reward` |


## RTX 4070 Run Defaults

Each config uses state observations, `model_size: 5`, `batch_size: 128`, `num_samples: 256`, `num_elites: 32`, and `num_pi_trajs: 12` to keep memory and wall-clock demand reasonable on an 8GB RTX 4070. Videos and agent checkpoint saving are disabled by default.

Step budgets are intentionally short enough for iteration but long enough to show meaningful learning trends:

| Environment | Steps | Eval frequency | Eval episodes |
|---|---:|---:|---:|
| `cartpole-swingup` | 75,000 | 7,500 | 5 |
| `finger-spin` | 100,000 | 10,000 | 5 |
| `cheetah-run` | 250,000 | 25,000 | 5 |
| `walker-walk` | 250,000 | 25,000 | 5 |


## Primary Seed 1 Commands

Run from `tdmpc2/`. These commands enable W&B and CSV logging.

```bash
python train.py --config-name configs/cartpole_swingup_baseline seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/cartpole_swingup_adaptive_only seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/cartpole_swingup_behavior_only seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/cartpole_swingup_full seed=1 enable_wandb=true save_csv=true

python train.py --config-name configs/finger_spin_baseline seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/finger_spin_adaptive_only seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/finger_spin_behavior_only seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/finger_spin_full seed=1 enable_wandb=true save_csv=true

python train.py --config-name configs/cheetah_run_baseline seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/cheetah_run_adaptive_only seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/cheetah_run_behavior_only seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/cheetah_run_full seed=1 enable_wandb=true save_csv=true

python train.py --config-name configs/walker_walk_baseline seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/walker_walk_adaptive_only seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/walker_walk_behavior_only seed=1 enable_wandb=true save_csv=true
python train.py --config-name configs/walker_walk_full seed=1 enable_wandb=true save_csv=true
```


## Save final point video
1. Set `save_agent` to `true`
2. Run
```Bash
python evaluate.py --config-name configs/cartpole_swingup_full \
  checkpoint=/home/anniehuang/WM/tdmpc2/tdmpc2/logs/cartpole-swingup/1/cartpole-swingup-full/models/final.pt \
  eval_episodes=1 save_video=true enable_wandb=false save_agent=false compile=false
```

## Metrics And Expected Plots

Use `eval/episode_reward` as the primary comparison. For mechanisms, inspect:

- `train/rollout_horizon` and `train/plan_rollout_horizon` for selected model-update and planning horizons.
- `train/horizon_at_h_min` and `train/horizon_at_h_max` for horizon collapse.
- `train/model_value_inconsistency` and `train/model_value_inconsistency_depth_*` for uncertainty-vs-depth behavior.
- `train/value_overestimation_proxy` for value exploitation pressure.
- `train/behavior_reg_loss` for the policy constraint term.

Expected plots:

1. Reward versus environment steps, grouped by environment and variant.
2. Selected horizon versus steps for adaptive variants.
3. Histogram or running mean of `horizon_at_h_min` and `horizon_at_h_max`.
4. Model-value inconsistency by depth versus steps.
5. Value overestimation proxy versus steps.
6. Behavior regularization loss versus steps for behavior-only and full variants.


## Interpretation Checklist

The full method is successful if it improves or matches baseline reward while showing evidence that it is using the new machinery rather than collapsing to a constant horizon. A strong result should show: higher early reward or final reward, lower `value_overestimation_proxy`, a non-degenerate selected-horizon distribution, and lower inconsistency at chosen rollout depths over training.

Ablation expectations:

- Baseline: stable fixed-horizon reference, but no horizon diagnostics beyond fixed values.
- Adaptive only: better horizon usage, possible remaining OOD value exploitation.
- Behavior only: lower exploitation pressure, but no dynamic rollout-depth control.
- Full: best reward/sample-efficiency tradeoff, especially on `finger-spin`, `cheetah-run`, and `walker-walk`.
